# Safe Trading — Análise do Backtest

**Estratégia:** EMA 8/21 Crossover + Filtro RSI + Confirmação de Volume  
**Mercado:** BTC/USDT 4H — Binance Spot

Este notebook carrega o resultado do backtest gerado por `scripts/backtest.py`  
e apresenta análises visuais e estatísticas detalhadas.

---
**Antes de rodar este notebook, execute:**
```bash
python scripts/backtest.py
```

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 1: Importar bibliotecas
# ─────────────────────────────────────────────────────────────
import os
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')

# Estilo dos gráficos: fundo escuro (combina com o tema do projeto)
plt.style.use('dark_background')

# Cores do projeto
COR_POSITIVO = '#3fb950'   # Verde — lucro, win
COR_NEGATIVO = '#f85149'   # Vermelho — perda, loss
COR_NEUTRO   = '#58a6ff'   # Azul — neutro, destaque
COR_TEXTO    = '#c9d1d9'   # Cinza claro — texto

print('✔ Bibliotecas carregadas com sucesso')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 2: Carregar resultado do backtest (JSON)
# ─────────────────────────────────────────────────────────────

# Caminho do arquivo JSON gerado pelo backtest
# Ajustar o caminho se necessário
PASTA_RAIZ   = os.path.dirname(os.getcwd())   # Um nível acima de notebooks/
ARQUIVO_JSON = os.path.join(PASTA_RAIZ, 'results', 'backtest_ema_resultado.json')

# Verificar se o arquivo existe
if not os.path.exists(ARQUIVO_JSON):
    raise FileNotFoundError(
        f'Arquivo não encontrado: {ARQUIVO_JSON}\n'
        f'Execute primeiro: python scripts/backtest.py'
    )

# Carregar o JSON
with open(ARQUIVO_JSON, 'r', encoding='utf-8') as f:
    resultado = json.load(f)

# Extrair as partes principais
metricas    = resultado['metricas']
lista_trades = resultado['trades']
parametros  = resultado['parametros']
periodo     = resultado['periodo']

print(f'✔ Backtest carregado com sucesso!')
print(f'  Estratégia: {resultado["estrategia"]}')
print(f'  Mercado:    {resultado["mercado"]}')
print(f'  Período:    {periodo["inicio"]} até {periodo["fim"]}')
print(f'  Trades:     {metricas.get("total_trades", 0)}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 3: Tabela de todos os trades
# ─────────────────────────────────────────────────────────────

# Converter lista de trades para DataFrame
df_trades = pd.DataFrame(lista_trades)

if df_trades.empty:
    print('Nenhum trade registrado.')
else:
    # Formatar datas
    df_trades['entrada_timestamp'] = pd.to_datetime(df_trades['entrada_timestamp'])
    df_trades['saida_timestamp']   = pd.to_datetime(df_trades['saida_timestamp'])

    # Selecionar e renomear colunas para exibição
    df_exibir = df_trades[[
        'entrada_timestamp', 'saida_timestamp',
        'preco_entrada', 'preco_saida',
        'quantidade_btc', 'pnl_usdt', 'retorno_pct',
        'resultado', 'motivo_saida', 'duracao_candles'
    ]].copy()

    df_exibir.columns = [
        'Entrada', 'Saída',
        'Preço Entrada ($)', 'Preço Saída ($)',
        'Qtd BTC', 'P&L (USDT)', 'Retorno (%)',
        'Resultado', 'Motivo Saída', 'Duração (candles)'
    ]

    # Formatar números
    df_exibir['Preço Entrada ($)'] = df_exibir['Preço Entrada ($)'].map('${:,.2f}'.format)
    df_exibir['Preço Saída ($)']   = df_exibir['Preço Saída ($)'].map('${:,.2f}'.format)
    df_exibir['P&L (USDT)']        = df_exibir['P&L (USDT)'].map('${:+,.2f}'.format)
    df_exibir['Retorno (%)']        = df_exibir['Retorno (%)'].map('{:+.2f}%'.format)

    # Exibir tabela
    pd.set_option('display.max_rows', 100)
    pd.set_option('display.width', 200)
    print(f'TABELA DE TODOS OS TRADES ({len(df_exibir)} trades):')
    print(df_exibir.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 4: Distribuição de retornos por trade (histograma)
# ─────────────────────────────────────────────────────────────

if not df_trades.empty:
    retornos = df_trades['retorno_pct'].values
    wins     = df_trades[df_trades['resultado'] == 'win']['retorno_pct'].values
    losses   = df_trades[df_trades['resultado'] == 'loss']['retorno_pct'].values

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('#0d1117')

    for ax in (ax1, ax2):
        ax.set_facecolor('#161b22')
        ax.tick_params(colors=COR_TEXTO)
        for spine in ax.spines.values():
            spine.set_color('#30363d')

    # Histograma geral de retornos
    bins = max(10, len(retornos) // 4)
    cores_hist = [COR_POSITIVO if r > 0 else COR_NEGATIVO for r in retornos]
    ax1.hist(retornos, bins=bins, color=COR_NEUTRO, alpha=0.7, edgecolor='#30363d')
    ax1.axvline(x=0, color='white', linestyle='--', linewidth=1, alpha=0.5)
    ax1.axvline(x=np.mean(retornos), color='#ffa657', linestyle='--',
                linewidth=1.5, alpha=0.9, label=f'Média: {np.mean(retornos):.2f}%')
    ax1.set_xlabel('Retorno por Trade (%)', color=COR_TEXTO)
    ax1.set_ylabel('Frequência', color=COR_TEXTO)
    ax1.set_title('Distribuição de Retornos', color='#e6edf3', fontweight='bold')
    ax1.legend(labelcolor=COR_TEXTO, facecolor='#21262d', edgecolor='#30363d')

    # Box plot: wins vs losses
    dados_box = []
    labels_box = []
    cores_box  = []
    if len(wins) > 0:
        dados_box.append(wins)
        labels_box.append(f'Wins\n(n={len(wins)})')
        cores_box.append(COR_POSITIVO)
    if len(losses) > 0:
        dados_box.append(losses)
        labels_box.append(f'Losses\n(n={len(losses)})')
        cores_box.append(COR_NEGATIVO)

    if dados_box:
        bp = ax2.boxplot(dados_box, labels=labels_box, patch_artist=True,
                         medianprops={'color': 'white', 'linewidth': 2},
                         whiskerprops={'color': '#6e7681'},
                         capprops={'color': '#6e7681'},
                         flierprops={'markerfacecolor': '#6e7681', 'markersize': 4})
        for patch, cor in zip(bp['boxes'], cores_box):
            patch.set_facecolor(cor)
            patch.set_alpha(0.6)
    ax2.axhline(y=0, color='white', linestyle='--', linewidth=1, alpha=0.5)
    ax2.set_ylabel('Retorno (%)', color=COR_TEXTO)
    ax2.set_title('Wins vs Losses', color='#e6edf3', fontweight='bold')

    plt.suptitle('Análise de Retornos por Trade', color='#e6edf3',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    print(f'Estatísticas de retorno:')
    print(f'  Média:    {np.mean(retornos):+.2f}%')
    print(f'  Mediana:  {np.median(retornos):+.2f}%')
    print(f'  Desvio:   {np.std(retornos):.2f}%')
    print(f'  Mínimo:   {np.min(retornos):+.2f}%')
    print(f'  Máximo:   {np.max(retornos):+.2f}%')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 5: Drawdown ao longo do tempo
# ─────────────────────────────────────────────────────────────

if not df_trades.empty:
    # Reconstruir curva de capital a partir dos trades
    capital_historico = [resultado['capital_inicial']]
    datas_historico   = [pd.to_datetime(df_trades['entrada_timestamp'].iloc[0])]

    for trade in lista_trades:
        capital_historico.append(trade['capital_apos'])
        datas_historico.append(pd.to_datetime(trade['saida_timestamp']))

    capital_series = pd.Series(capital_historico, index=datas_historico)

    # Calcular drawdown a cada ponto
    pico_corrente = capital_series.cummax()
    drawdown      = (capital_series - pico_corrente) / pico_corrente * 100

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    fig.patch.set_facecolor('#0d1117')

    for ax in (ax1, ax2):
        ax.set_facecolor('#161b22')
        ax.tick_params(colors=COR_TEXTO)
        for spine in ax.spines.values():
            spine.set_color('#30363d')

    # Curva de equity
    ax1.plot(capital_series.index, capital_series.values,
             color=COR_NEUTRO, linewidth=1.5, label='Capital')
    ax1.axhline(y=resultado['capital_inicial'], color='#6e7681',
                linestyle='--', linewidth=0.8, alpha=0.7, label='Capital Inicial')
    ax1.fill_between(capital_series.index, capital_series.values, resultado['capital_inicial'],
                     where=(capital_series >= resultado['capital_inicial']),
                     alpha=0.15, color=COR_POSITIVO)
    ax1.fill_between(capital_series.index, capital_series.values, resultado['capital_inicial'],
                     where=(capital_series < resultado['capital_inicial']),
                     alpha=0.15, color=COR_NEGATIVO)
    ax1.set_ylabel('Capital (USDT)', color=COR_TEXTO)
    ax1.set_title('Curva de Equity e Drawdown', color='#e6edf3',
                  fontweight='bold', fontsize=12)
    ax1.legend(labelcolor=COR_TEXTO, facecolor='#21262d', edgecolor='#30363d')
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

    # Drawdown
    ax2.fill_between(drawdown.index, drawdown.values, 0,
                     alpha=0.6, color=COR_NEGATIVO, label='Drawdown')
    ax2.plot(drawdown.index, drawdown.values, color=COR_NEGATIVO, linewidth=0.8)
    ax2.axhline(y=-metricas.get('max_drawdown_pct', 0),
                color='#ffa657', linestyle='--', linewidth=1,
                label=f'Max DD: -{metricas.get("max_drawdown_pct", 0):.1f}%')
    ax2.set_ylabel('Drawdown (%)', color=COR_TEXTO)
    ax2.set_xlabel('Data', color=COR_TEXTO)
    ax2.legend(labelcolor=COR_TEXTO, facecolor='#21262d', edgecolor='#30363d')
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))

    plt.tight_layout()
    plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 6: Tabela de métricas formatada
# ─────────────────────────────────────────────────────────────

print('=' * 55)
print('  MÉTRICAS DO BACKTEST — RESUMO COMPLETO')
print('=' * 55)

# Organizar métricas em grupos
grupos = {
    'CAPITAL': [
        ('Capital Inicial',       f"${metricas.get('capital_inicial', 0):,.2f}"),
        ('Capital Final',         f"${metricas.get('capital_final', 0):,.2f}"),
        ('Retorno Total',         f"{metricas.get('retorno_total_pct', 0):+.2f}%"),
    ],
    'TRADES': [
        ('Total de Trades',       str(metricas.get('total_trades', 0))),
        ('Trades Vencedores',     str(metricas.get('total_vencedores', 0))),
        ('Trades Perdedores',     str(metricas.get('total_perdedores', 0))),
        ('Win Rate',              f"{metricas.get('win_rate_pct', 0):.1f}%"),
    ],
    'QUALIDADE': [
        ('Profit Factor',         f"{metricas.get('profit_factor', 0):.4f}"),
        ('Sharpe Ratio',          f"{metricas.get('sharpe_ratio', 0):.4f}"),
        ('Max Drawdown',          f"{metricas.get('max_drawdown_pct', 0):.2f}%"),
    ],
    'POR TRADE': [
        ('Média Ganho/Trade',     f"{metricas.get('media_ganho_pct', 0):+.2f}%"),
        ('Média Perda/Trade',     f"{metricas.get('media_perda_pct', 0):+.2f}%"),
        ('Duração Média',         f"{metricas.get('duracao_media_candles', 0):.1f} candles ({metricas.get('duracao_media_candles', 0) * 4:.0f}h)"),
    ],
}

for grupo, itens in grupos.items():
    print(f'\n  {grupo}:')
    for nome, valor in itens:
        print(f'    {nome:<25} {valor:>15}')

print('\n' + '=' * 55)

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 7: Gráfico de motivos de saída
# ─────────────────────────────────────────────────────────────

if not df_trades.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    fig.patch.set_facecolor('#0d1117')

    for ax in (ax1, ax2):
        ax.set_facecolor('#161b22')
        ax.tick_params(colors=COR_TEXTO)
        for spine in ax.spines.values():
            spine.set_color('#30363d')

    # Gráfico 1: Pizza de motivos de saída
    motivos = df_trades['motivo_saida'].value_counts()
    cores_pizza = ['#3fb950', '#f85149', '#79c0ff', '#ffa657', '#d2a8ff']
    wedges, texts, autotexts = ax1.pie(
        motivos.values,
        labels=motivos.index,
        autopct='%1.1f%%',
        colors=cores_pizza[:len(motivos)],
        startangle=90
    )
    for text in texts + autotexts:
        text.set_color(COR_TEXTO)
        text.set_fontsize(9)
    ax1.set_title('Motivos de Saída', color='#e6edf3', fontweight='bold')

    # Gráfico 2: Barras de P&L por motivo de saída
    pnl_por_motivo = df_trades.groupby('motivo_saida')['pnl_usdt'].sum()
    cores_barras = [COR_POSITIVO if v > 0 else COR_NEGATIVO for v in pnl_por_motivo.values]
    barras = ax2.bar(pnl_por_motivo.index, pnl_por_motivo.values,
                     color=cores_barras, alpha=0.8, edgecolor='#30363d')
    ax2.axhline(y=0, color='white', linewidth=0.8, alpha=0.5)
    ax2.set_ylabel('P&L Total (USDT)', color=COR_TEXTO)
    ax2.set_title('P&L Total por Motivo de Saída', color='#e6edf3', fontweight='bold')
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:+,.0f}'))

    # Adicionar valores nas barras
    for barra in barras:
        altura = barra.get_height()
        ax2.text(
            barra.get_x() + barra.get_width() / 2,
            altura + (max(pnl_por_motivo.values) * 0.02),
            f'${altura:+,.2f}',
            ha='center', va='bottom',
            color=COR_TEXTO, fontsize=8
        )

    plt.tight_layout()
    plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# CÉLULA 8: CONCLUSÃO — A estratégia passa nos critérios?
# ─────────────────────────────────────────────────────────────

# Critérios mínimos definidos na especificação da estratégia
CRITERIOS = {
    'Sharpe Ratio > 1.0':  ('sharpe_ratio',    lambda v: v > 1.0,   '>1.0',   '>1.5'),
    'Max Drawdown < 20%':  ('max_drawdown_pct', lambda v: v < 20.0,  '<20%',   '<15%'),
    'Profit Factor > 1.5': ('profit_factor',    lambda v: v > 1.5,   '>1.5',   '>1.8'),
    'Win Rate > 40%':      ('win_rate_pct',     lambda v: v > 40.0,  '>40%',   '>48%'),
    'Total Trades > 30':   ('total_trades',     lambda v: v > 30,    '>30',    '>50'),
}

print('=' * 65)
print('  CONCLUSÃO — A ESTRATÉGIA PASSA NOS CRITÉRIOS MÍNIMOS?')
print('=' * 65)
print(f'  {"Critério":<30} {"Valor":>10}  {"Meta Min":>8}  {"Meta Ideal":>10}  Status')
print(f'  {"-"*30} {"-"*10}  {"-"*8}  {"-"*10}  {"------"}')

todos_aprovados = True
for nome_criterio, (chave, condicao, meta_min, meta_ideal) in CRITERIOS.items():
    valor = metricas.get(chave, 0)
    passou = condicao(valor)
    if not passou:
        todos_aprovados = False

    # Formatar valor
    if 'pct' in chave or 'rate' in chave:
        valor_fmt = f'{valor:.2f}%'
    elif isinstance(valor, int):
        valor_fmt = str(valor)
    else:
        valor_fmt = f'{valor:.4f}'

    status = '✔ PASSOU' if passou else '✘ FALHOU'
    print(f'  {nome_criterio:<30} {valor_fmt:>10}  {meta_min:>8}  {meta_ideal:>10}  {status}')

print('=' * 65)

if todos_aprovados:
    print('\n  ✔ ESTRATÉGIA APROVADA')
    print('  A estratégia atendeu todos os critérios mínimos.')
    print('  Próximo passo: F3 — Implementar no Freqtrade em modo dry-run.')
else:
    print('\n  ✘ ESTRATÉGIA REPROVADA')
    print('  Alguns critérios não foram atingidos.')
    print('  Próximo passo: Ajustar parâmetros e rodar novo backtest.')
    print('  Sugestões:')
    sharpe = metricas.get('sharpe_ratio', 0)
    if sharpe < 1.0:
        print('    - Sharpe baixo: considerar filtros adicionais ou ajustar RSI_MIN/MAX')
    dd = metricas.get('max_drawdown_pct', 0)
    if dd >= 20.0:
        print('    - Drawdown alto: considerar reduzir RISCO_POR_TRADE ou adicionar stop diário')
    pf = metricas.get('profit_factor', 0)
    if pf < 1.5:
        print('    - Profit Factor baixo: aumentar o multiplicador de TP ou refinar entradas')

print('\n  Período analisado:', periodo['inicio'], 'até', periodo['fim'])
print('=' * 65)